# RSNA Knee - 5-fold CV fusion ensemble submission

Raptor v5 CoAtNet + RadImageNet ResNet-50 residual fusion. Folds [0, 1, 2, 3, 4]: `fusion_best.pt` of each fold
(selected by validation accuracy = mean of gold74 and left-out-fold macro AUC). Prediction = mean of the
fold models' sigmoid probabilities; no TTA. Same six-slot preprocessing and same-series windows as training.

Embedded source digests: `{"models/__init__.py": "360790aae4faf61c", "models/raptor_v5_radimagenet.py": "56ea1360da1a3958", "preprocess_six_slot_raptor.py": "cc5e92682fd0b691", "infer_fusion_cv5_submission.py": "c0563fcbd098cf94"}`


In [ ]:
from pathlib import Path
import sys
SOURCES = {'models/__init__.py': '"""Trainable models for RSNA knee experiments."""\n', 'models/raptor_v5_radimagenet.py': '"""Raptor v5 with residual RadImageNet fusion inside CoAtNet.\n\nInput is an already preprocessed [B, K, 3, 384, 384] triplet bag. Both\nencoders receive exactly the same tensor; preprocessing belongs to the loader.\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Mapping\nfrom contextlib import contextmanager, nullcontext\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport torch\nfrom torch import Tensor, nn\nfrom torch.utils.checkpoint import checkpoint\n\n\nCOATNET_ARCH = "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"\nLABELS = (\n    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",\n    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker\'s",\n    "Contusion", "Fracture",\n)\n\n\nRAD_INPUTS = ("shared", "pm1_triplet", "pm1_middle")\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\n\n@dataclass(frozen=True)\nclass FusionConfig:\n    stage2_hidden: int = 1024\n    stage4_hidden: int = 2048\n    mlp_dropout: float = 0.0\n    head_dropout: float = 0.2\n    gradient_checkpointing: bool = False\n    # What the RadImageNet trunk receives (CoAtNet always gets the ImageNet-normalized triplet):\n    # \'shared\' = the same triplet (original behaviour and the default for older checkpoints);\n    # \'pm1_triplet\' = the three slices rescaled to [-1, 1];\n    # \'pm1_middle\' = the middle slice in [-1, 1] copied to 3 channels (baseline Rad convention).\n    rad_input: str = "shared"\n\n\ndef rad_input_transform(image: Tensor, mode: str) -> Tensor:\n    """Map an ImageNet-normalized triplet [N,3,H,W] to the RadImageNet input convention."""\n    if mode == "shared":\n        return image\n    if mode not in RAD_INPUTS:\n        raise ValueError(f"Unknown rad_input {mode!r}; expected one of {RAD_INPUTS}")\n    mean = torch.tensor(IMAGENET_MEAN, device=image.device, dtype=image.dtype).view(1, 3, 1, 1)\n    std = torch.tensor(IMAGENET_STD, device=image.device, dtype=image.dtype).view(1, 3, 1, 1)\n    pm1 = (image*std + mean)*2 - 1          # exact inverse of the ImageNet normalization, then [-1, 1]\n    if mode == "pm1_middle":\n        pm1 = pm1[:, 1:2].expand(-1, 3, -1, -1)\n    return pm1.contiguous()\n\n\n@contextmanager\ndef preserve_batchnorm_buffers(module: nn.Module):\n    """Restore running statistics after a probe or checkpoint recomputation.\n\n    BatchNorm, SyncBatchNorm and timm BatchNormAct subclasses keep their normal\n    training behavior (including synchronization and trainable affine weights).\n    Only running_mean, running_var and num_batches_tracked are restored, so a\n    recomputed forward contributes gradients but does not count as a second\n    observation. Enter this context for recomputation only, after the original\n    forward has updated the statistics.\n    """\n    saved = [\n        (buffer, buffer.detach().clone())\n        for layer in module.modules()\n        if isinstance(layer, nn.modules.batchnorm._BatchNorm)\n        for name in ("running_mean", "running_var", "num_batches_tracked")\n        if (buffer := getattr(layer, name, None)) is not None\n    ]\n    try:\n        yield\n    finally:\n        with torch.no_grad():\n            for buffer, value in saved:\n                buffer.copy_(value)\n\n\nclass ConcatenationMLP(nn.Module):\n    """Three affine layers with GELU, applied to each spatial position.\n\n    Concatenation precedes every learned operation. Only the output projection\n    is zero-initialized, so the residual is exactly zero without a dead MLP.\n    """\n\n    def __init__(self, first_channels: int, second_channels: int,\n                 out_channels: int, hidden: int, dropout: float = 0.0):\n        super().__init__()\n        if min(first_channels, second_channels, out_channels, hidden) <= 0:\n            raise ValueError("MLP channel dimensions must be positive")\n        self.first_channels = first_channels\n        self.second_channels = second_channels\n        self.mlp = nn.Sequential(\n            nn.LayerNorm(first_channels + second_channels),\n            nn.Linear(first_channels + second_channels, hidden),\n            nn.GELU(), nn.Dropout(dropout),\n            nn.Linear(hidden, hidden),\n            nn.GELU(), nn.Dropout(dropout),\n            nn.Linear(hidden, out_channels),\n        )\n        nn.init.zeros_(self.mlp[-1].weight)\n        nn.init.zeros_(self.mlp[-1].bias)\n\n    def forward(self, first: Tensor, second: Tensor) -> Tensor:\n        if (first.ndim != 4 or second.ndim != 4\n                or first.shape[0] != second.shape[0]\n                or first.shape[2:] != second.shape[2:]\n                or first.shape[1] != self.first_channels\n                or second.shape[1] != self.second_channels):\n            raise ValueError(f"Incompatible fusion features: {first.shape}, {second.shape}")\n        joined = torch.cat((first, second), dim=1).permute(0, 2, 3, 1)\n        return self.mlp(joined).permute(0, 3, 1, 2).contiguous()\n\n\nclass RadImageNetEncoder(nn.Module):\n    """The same torchvision ResNet-50 trunk used by the baseline Rad encoder.\n\n    Keep `backbone.0` ... `backbone.7` names for strict ResNet50.pt loading.\n    conv3_x is backbone[5] (torchvision layer2); conv5_x is backbone[7]\n    (layer4). No ImageNet weights are substituted for RadImageNet weights.\n    """\n\n    def __init__(self):\n        super().__init__()\n        from torchvision.models import resnet50\n\n        self.backbone = nn.Sequential(*list(resnet50(weights=None).children())[:-2])\n\n    def forward(self, image: Tensor) -> tuple[Tensor, Tensor]:\n        x = image\n        conv3 = None\n        for index, layer in enumerate(self.backbone):\n            x = layer(x)\n            if index == 5:\n                conv3 = x\n        return conv3, x\n\n\nclass FusedCoAtNet(nn.Module):\n    """Stage2 Z += F1(X,Z), Stage4 V += F2(V,Y), then normal CoAtNet pooling."""\n\n    def __init__(self, coatnet: nn.Module, radimagenet: nn.Module,\n                 config: FusionConfig = FusionConfig()):\n        super().__init__()\n        self.coatnet = coatnet\n        self.radimagenet = radimagenet\n        self.num_features = coatnet.num_features\n        channels = [entry["num_chs"] for entry in coatnet.feature_info]\n        if len(coatnet.stages) != 4 or channels[-4:] != [128, 256, 512, 1024]:\n            raise ValueError("Expected the four-stage Raptor v5 CoAtNet RMLP-2")\n        self.f1 = ConcatenationMLP(512, 256, 256, config.stage2_hidden, config.mlp_dropout)\n        self.f2 = ConcatenationMLP(1024, 2048, 1024, config.stage4_hidden, config.mlp_dropout)\n        self.gradient_checkpointing = config.gradient_checkpointing\n        if config.rad_input not in RAD_INPUTS:\n            raise ValueError(f"Unknown rad_input {config.rad_input!r}; expected one of {RAD_INPUTS}")\n        self.rad_input = config.rad_input\n        self.checkpoint_parts: frozenset[str] = frozenset()\n        self.requires_grad_(True)\n        self.coatnet.set_grad_checkpointing(config.gradient_checkpointing)\n\n    CHECKPOINT_PARTS = ("rad", "s1", "s2", "s3", "s4", "fusion")\n\n    def set_checkpoint_parts(self, parts) -> None:\n        """Runtime-only activation recomputation for selected whole sections.\n\n        Parts: rad (ResNet-50), s1 (stem+S1), s2, s3, s4 (CoAtNet stages) and\n        fusion (both MLPs). Not stored in checkpoints; numerics are unchanged.\n        These whole-section checkpoints preserve BatchNorm running buffers\n        during recomputation. This does not cover timm\'s separate internal\n        checkpointing; leave config.gradient_checkpointing=False when using\n        trainable BatchNorm statistics with this path.\n        """\n        parts = frozenset(parts)\n        unknown = parts - set(self.CHECKPOINT_PARTS)\n        if unknown:\n            raise ValueError(f"Unknown checkpoint parts: {sorted(unknown)}")\n        self.checkpoint_parts = parts\n\n    def _run(self, part: str, fn, *args):\n        if part in self.checkpoint_parts and self.training and torch.is_grad_enabled():\n            # s1 is a bound method spanning stem and stage 1; the other sections\n            # are individual modules. Fusion MLPs have no BatchNorm buffers.\n            scope = fn if isinstance(fn, nn.Module) else self.coatnet\n            return checkpoint(\n                fn, *args, use_reentrant=False,\n                context_fn=lambda: (nullcontext(), preserve_batchnorm_buffers(scope)),\n            )\n        return fn(*args)\n\n    def _stem_s1(self, image: Tensor) -> Tensor:\n        return self.coatnet.stages[0](self.coatnet.stem(image))\n\n    def _fuse(self, mlp: nn.Module, first: Tensor, second: Tensor) -> Tensor:\n        if ((self.gradient_checkpointing or "fusion" in self.checkpoint_parts)\n                and self.training and torch.is_grad_enabled()):\n            return checkpoint(mlp, first, second, use_reentrant=False)\n        return mlp(first, second)\n\n    def forward_features(self, image: Tensor) -> Tensor:\n        if image.ndim != 4 or tuple(image.shape[1:]) != (3, 384, 384):\n            raise ValueError("Both backbones require [N, 3, 384, 384] preprocessed triplets")\n        # CoAtNet gets the triplet as saved; RadImageNet gets it in its configured convention.\n        x, y = self._run("rad", self.radimagenet, rad_input_transform(image, self.rad_input))\n        z = self._run("s1", self._stem_s1, image)\n        z = self._run("s2", self.coatnet.stages[1], z)  # S2 convolution, [N,256,48,48]\n        z = z + self._fuse(self.f1, x, z)  # F1 input order is (X,Z)\n        v = self._run("s3", self.coatnet.stages[2], z)  # S3 receives the UPDATED Z\n        v = self._run("s4", self.coatnet.stages[3], v)  # S4 transformer, [N,1024,12,12]\n        v = v + self._fuse(self.f2, v, y)  # F2 input order is (V,Y)\n        return self.coatnet.norm(v)\n\n    def forward(self, image: Tensor) -> Tensor:\n        return self.coatnet.forward_head(self.forward_features(image))\n\n\nclass RaptorClassifier(nn.Module):\n    """Original Raptor attention-MIL head, retaining checkpoint parameter names."""\n\n    def __init__(self, backbone: nn.Module, F_dim: int, n: int = 12, drop: float = 0.2):\n        super().__init__()\n        self.backbone = backbone\n        self.norm = nn.LayerNorm(F_dim)\n        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))\n        self.clsW = nn.Parameter(torch.zeros(n, F_dim))\n        self.clsb = nn.Parameter(torch.zeros(n))\n        nn.init.trunc_normal_(self.clsW, std=0.02)\n        self.n = n\n\n    def encode(self, x: Tensor) -> Tensor:\n        if x.ndim != 5 or x.shape[0] == 0 or x.shape[1] == 0:\n            raise ValueError("Raptor expects a nonempty [B,K,3,384,384] triplet bag")\n        b, k = x.shape[:2]\n        return self.backbone(x.flatten(0, 1)).reshape(b, k, -1)\n\n    def head(self, feats: Tensor) -> Tensor:\n        h = self.norm(feats)\n        a = torch.softmax(self.att(h), dim=1)\n        pooled = torch.einsum("bkn,bkf->bnf", a, h)\n        return (pooled * self.clsW).sum(-1) + self.clsb\n\n    def forward(self, x: Tensor) -> Tensor:\n        return self.head(self.encode(x))\n\n\ndef _checkpoint(source: str | Path | Mapping[str, Any]) -> Mapping[str, Any]:\n    return source if isinstance(source, Mapping) else torch.load(\n        source, map_location="cpu", weights_only=True,\n    )\n\n\ndef build_raptor_v5_radimagenet(\n    raptor_checkpoint: str | Path | Mapping[str, Any],\n    radimagenet_checkpoint: str | Path | Mapping[str, Any],\n    config: FusionConfig = FusionConfig(),\n) -> RaptorClassifier:\n    """Strict-load both pretrained parents, add zero residuals, unfreeze all.\n\n    Raptor checkpoint: {\'model\': state_dict, \'arch\': ..., \'res\': 384}.\n    Rad checkpoint: the baseline ResNet50.pt state dict, or {\'state_dict\': ...}.\n    Load and resume a *fused* checkpoint using load_fused_checkpoint instead.\n    """\n    import timm\n\n    ck = _checkpoint(raptor_checkpoint)\n    if ck.get("arch", COATNET_ARCH) != COATNET_ARCH or int(ck.get("res", 384)) != 384:\n        raise ValueError("Parent checkpoint must be Raptor v5 CoAtNet RMLP-2 at res=384")\n    coatnet = timm.create_model(COATNET_ARCH, pretrained=False, num_classes=0, in_chans=3, global_pool="avg")\n    model = RaptorClassifier(coatnet, F_dim=coatnet.num_features, drop=config.head_dropout)\n    model.load_state_dict(ck["model"], strict=True)\n    rad = RadImageNetEncoder()\n    rad_ck = _checkpoint(radimagenet_checkpoint)\n    rad.load_state_dict(rad_ck.get("state_dict", rad_ck), strict=True)\n    model.backbone = FusedCoAtNet(coatnet, rad, config)\n    model.fusion_config = config\n    model.requires_grad_(True)\n    return model\n\n\ndef optimizer_groups(model: RaptorClassifier, backbone_lr: float = 1e-5,\n                     fusion_lr: float = 1e-4, head_lr: float = 1e-4) -> list[dict[str, Any]]:\n    """All parameters, exactly once, with both backbones unfrozen."""\n    if not isinstance(model.backbone, FusedCoAtNet):\n        raise TypeError("Expected the fused Raptor model")\n    if any(not p.requires_grad for p in model.parameters()):\n        raise ValueError("Both backbones and the fusion/head must remain unfrozen")\n    groups = {"coatnet": [], "radimagenet": [], "fusion": [], "head": []}\n    for name, parameter in model.named_parameters():\n        if name.startswith("backbone.coatnet."):\n            group = "coatnet"\n        elif name.startswith("backbone.radimagenet."):\n            group = "radimagenet"\n        elif name.startswith(("backbone.f1.", "backbone.f2.")):\n            group = "fusion"\n        else:\n            group = "head"\n        groups[group].append(parameter)\n    rates = dict(coatnet=backbone_lr, radimagenet=backbone_lr, fusion=fusion_lr, head=head_lr)\n    return [{"name": name, "params": parameters, "lr": rates[name]}\n            for name, parameters in groups.items()]\n\n\ndef fused_checkpoint(model: RaptorClassifier) -> dict[str, Any]:\n    """Serialize model/config; trainers can add optimizer, scaler and epoch.\n\n    Tensors are copied to CPU: state_dict() alone shares storage with the live parameters, so\n    swapping weights back afterwards (e.g. EMA -> raw) would silently change the checkpoint.\n    """\n    state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}\n    return {"format": "raptor_v5_radimagenet_v1", "arch": COATNET_ARCH, "res": 384,\n            "labels": list(LABELS), "fusion_config": asdict(model.fusion_config),\n            "model": state}\n\n\ndef load_fused_checkpoint(source: str | Path | Mapping[str, Any]) -> RaptorClassifier:\n    import timm\n\n    ck = _checkpoint(source)\n    if (ck.get("format") != "raptor_v5_radimagenet_v1" or ck.get("arch") != COATNET_ARCH\n            or ck.get("res") != 384 or ck.get("labels") != list(LABELS)):\n        raise ValueError("Unsupported fused checkpoint architecture/input/label contract")\n    config = FusionConfig(**ck["fusion_config"])\n    coatnet = timm.create_model(COATNET_ARCH, pretrained=False, num_classes=0, in_chans=3, global_pool="avg")\n    backbone = FusedCoAtNet(coatnet, RadImageNetEncoder(), config)\n    model = RaptorClassifier(backbone, F_dim=backbone.num_features, drop=config.head_dropout)\n    model.fusion_config = config\n    model.load_state_dict(ck["model"], strict=True)\n    model.requires_grad_(True)\n    return model\n', 'preprocess_six_slot_raptor.py': '"""Six MRI slots with the saved Raptor V5 image preprocessing contract.\n\nRuns on Kaggle. Outputs fully normalized float16 [K,3,384,384] study bags.\nNo DICOM data is downloaded to the local development computer.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport io\nimport json\nimport os\nimport re\nimport shutil\nimport threading\nimport time\nimport zipfile\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport pydicom\nimport torch\nimport torch.nn.functional as F\nfrom pydicom.pixels import apply_modality_lut\n\n# Six-slot extension; original V5 used five slots with 18/14/12/8/12 references.\nSLOTS = [\n    ("SAG_FLUID_FS", "Sagittal", 1, 1, 18),\n    ("COR_FLUID_FS", "Coronal", 1, 1, 12),\n    ("AX_FLUID_FS", "Axial", 1, 1, 12),\n    ("SAG_FLUID_NOFS", "Sagittal", 1, 0, 14),\n    ("COR_T1", "Coronal", 0, 0, 8),\n    ("SAG_T1", "Sagittal", 0, 0, 8),\n]\nMEAN = torch.tensor([.485, .456, .406]).view(1, 3, 1, 1)\nSTD = torch.tensor([.229, .224, .225]).view(1, 3, 1, 1)\nLABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",\n          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\nRECIPE = {\n    "format": "rsna_six_slot_raptor_v5_v1", "slots": SLOTS,\n    "source_span": [.02, .98], "intensity_percentiles": [2, 98],\n    "intensity_reference": "original evenly spaced per-series reference slices",\n    "crop_mm": 140, "cache_resolution": 336, "model_resolution": 384,\n    "resize": "cv2.INTER_AREA to 336; torch bilinear align_corners=False to 384",\n    "quantization": "uint8 after 336 resize, before /255 and 384 interpolation",\n    "mean": [.485, .456, .406], "std": [.229, .224, .225],\n    "dtype": "float16", "normalized": True, "source_budget": 64,\n    "triplets": "sliding adjacent sampled slices in concatenated slot order, like parent V5",\n    "missing_slots": "absent, never substitute a different category or fabricate pixels",\n    "selector": "dicom_weight_type; positive fat-sat header evidence overrides metadata",\n    "labels": "annotations are separate from pixels; gold74 excluded from training split",\n}\nRECIPE_SHA = hashlib.sha256(json.dumps(RECIPE, sort_keys=True).encode()).hexdigest()\nLOCK = threading.Lock()\n\n\ndef json_write(path, value):\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(value, indent=2, allow_nan=False) + "\\n")\n    tmp.replace(path)\n\n\ndef allocate(capacities, weights, target=64):\n    capacities = np.asarray(capacities, dtype=np.int64)\n    weights = np.asarray(weights, dtype=np.float64)\n    ideal = target * weights / weights.sum()\n    quotas = np.floor(ideal).astype(np.int64)\n    for i in np.argsort(-(ideal-quotas), kind="stable")[:target-int(quotas.sum())]:\n        quotas[i] += 1\n    quotas = np.minimum(quotas, capacities)\n    deficit = np.zeros(len(quotas))\n    wanted = min(target, int(capacities.sum()))\n    while quotas.sum() < wanted:\n        available = quotas < capacities\n        active = np.where(available, weights, 0)\n        deficit[available] += active[available]/active.sum()\n        choice = int(np.argmax(np.where(available, deficit, -np.inf)))\n        quotas[choice] += 1\n        deficit[choice] -= 1\n    return quotas\n\n\ndef flag(value):\n    try:\n        value = float(value)\n        return int(value) if value in (0, 1) else None\n    except (ValueError, TypeError):\n        return None\n\n\ndef weight_type(h):\n    desc = re.sub(r"[_\\-.]", " ", f"{getattr(h, \'SeriesDescription\', \'\')} {getattr(h, \'SequenceName\', \'\')}".lower())\n    t1 = bool(re.search(r"\\bt1\\b|\\bt1w\\b", desc))\n    t2 = bool(re.search(r"\\bt2\\b|\\bt2w\\b", desc))\n    pdw = bool(re.search(r"\\bpd\\b|\\bpdw\\b|proton|\\bdp\\b|dens", desc))\n    tr, te = float(getattr(h, \'RepetitionTime\', \'nan\')), float(getattr(h, \'EchoTime\', \'nan\'))\n    if t1 and not t2 and not pdw:\n        return "T1"\n    if t2 and not pdw:\n        return "T2"\n    if pdw:\n        return "PD"\n    if "GR" in str(getattr(h, "ScanningSequence", "")).upper():\n        return "GRE"\n    return "T1" if tr < 800 else "T2" if te > 60 else "PD" if tr >= 800 else "UNK"\n\n\ndef classify(row, directory):\n    files = sorted(directory.glob("*.dcm"))\n    if not files:\n        return None\n    h = pydicom.dcmread(files[len(files)//2], stop_before_pixels=True)\n    kind = weight_type(h)\n    # In this competition CSV, Fluid_Sensitive and Fat_Suppression are\n    # effectively coupled. PD/T2 without FS must be recovered from the header,\n    # as in the original recovered six-slot classifier.\n    fluid = int(kind in ("PD", "T2")) if kind != "UNK" else flag(row.get("Fluid_Sensitive"))\n    fs = flag(row.get("Fat_Suppression"))\n    desc = re.sub(r"[_\\-.]", " ", f"{getattr(h, \'SeriesDescription\', \'\')} {getattr(h, \'SequenceName\', \'\')}".lower())\n    options = getattr(h, "ScanOptions", [])\n    options = options.split(\'\\\\\') if isinstance(options, str) else options\n    opts = set(str(v).upper() for v in options)\n    header_fs = bool(re.search(r"\\bfs\\b|fat\\s*sat|\\bstir\\b|\\bspair\\b|\\bspir\\b|\\bwe\\b|water excit|\\btirm\\b|\\bsting\\b|\\bfatsup\\b|smart fat|\\bwater\\b", desc)) or bool(opts & {"FS", "FATSAT", "FAT_SAT", "FSAT"})\n    if header_fs:\n        fs = 1\n    elif fs is None:\n        fs = 0\n    return {"uid": str(row["SeriesInstanceUID"]), "plane": row["Anatomical_Plane"],\n            "fluid": fluid, "fs": fs, "weight": kind, "files": files,\n            "description": str(getattr(h, "SeriesDescription", ""))}\n\n\ndef order_files(series):\n    records, sop_uids = [], set()\n    for file in series["files"]:\n        h = pydicom.dcmread(file, stop_before_pixels=True)\n        sop = str(h.SOPInstanceUID)\n        if sop in sop_uids:\n            raise ValueError(f"Duplicate SOPInstanceUID: {file}")\n        sop_uids.add(sop)\n        if str(h.SeriesInstanceUID) != series["uid"]:\n            raise ValueError(f"Series UID mismatch: {file}")\n        iop, ipp = getattr(h, "ImageOrientationPatient", None), getattr(h, "ImagePositionPatient", None)\n        if iop is not None and ipp is not None and len(iop) == 6:\n            position = float(np.dot(np.asarray(ipp, float), np.cross(np.asarray(iop[:3], float), np.asarray(iop[3:], float))))\n        else:\n            position = float(h.InstanceNumber)\n        if not np.isfinite(position):\n            raise ValueError(f"Nonfinite slice coordinate: {file}")\n        spacing = float(getattr(h, "PixelSpacing", [.5])[0])\n        records.append((position, str(file), spacing))\n    records.sort(key=lambda r: (r[0], r[1]))\n    good_spacing = [r[2] for r in records if np.isfinite(r[2]) and r[2] > 0]\n    median = float(np.median(good_spacing)) if good_spacing else .5\n    return [(file, spacing if np.isfinite(spacing) and spacing > 0 else median) for _, file, spacing in records]\n\n\ndef pixels(file):\n    d = pydicom.dcmread(file)\n    a = apply_modality_lut(d.pixel_array, d).astype(np.float32)\n    if a.ndim != 2 or not np.isfinite(a).all():\n        raise ValueError(f"Invalid acquired pixels: {file}")\n    if str(getattr(d, "PhotometricInterpretation", "")) == "MONOCHROME1":\n        a = a.max()-a\n    return a\n\n\ndef crop_plane(a, spacing, low, high):\n    a = np.clip((a-low)/(high-low+1e-6), 0, 1)\n    h, w = a.shape\n    size = min(int(round(140/max(spacing, .001))), min(h, w))\n    y, x = (h-size)//2, (w-size)//2\n    a = cv2.resize(a[y:y+size, x:x+size], (336, 336), interpolation=cv2.INTER_AREA)\n    return (a*255).astype(np.uint8)\n\n\ndef model_ready(planes):\n    # Preserve parent\'s /255, interpolation, then channel-specific normalization.\n    volume = np.stack(planes)\n    source = torch.from_numpy(volume.astype(np.float32)/255)\n    resized = F.interpolate(source[:, None], size=(384, 384), mode="bilinear", align_corners=False)[:, 0]\n    x = torch.stack([resized[:-2], resized[1:-1], resized[2:]], dim=1)\n    return ((x-MEAN)/STD).numpy().astype(np.float16)\n\n\ndef prepare(uid, rows, series_root):\n    candidates, warnings = [], []\n    for row in rows:\n        s = classify(row, series_root/uid/str(row["SeriesInstanceUID"]))\n        if s is not None:\n            candidates.append(s)\n    used, pools, selection = set(), [], []\n    for name, plane, fluid, fs, reference_count in SLOTS:\n        matching = [s for s in candidates if s["uid"] not in used and s["plane"] == plane\n                    and s["fluid"] == fluid and s["fs"] == fs\n                    and (fluid == 1 or s["weight"] == "T1")]\n        # Retain metadata row priority, matching the Raptor selection convention.\n        if not matching:\n            pools.append(None)\n            selection.append({"slot": name, "present": False})\n            continue\n        series = matching[0]\n        used.add(series["uid"])\n        files = order_files(series)\n        lo = int(len(files)*.02)\n        hi = max(int(len(files)*.98)-1, lo)\n        refs = np.linspace(lo, hi, reference_count).round().astype(int)\n        pools.append((files, lo, hi, refs))\n        selection.append({"slot": name, "present": True, "series_uid": series["uid"],\n                          "description": series["description"], "weight": series["weight"]})\n    capacities = [0 if p is None else p[2]-p[1]+1 for p in pools]\n    quotas = allocate(capacities, [s[4] for s in SLOTS])\n    planes, slot_ids, sources, intensity = [], [], [], []\n    for slot, (pool, count) in enumerate(zip(pools, quotas)):\n        if pool is None or not count:\n            continue\n        files, lo, hi, refs = pool\n        picks = lo + np.linspace(0, hi-lo, int(count)).round().astype(int)\n        arrays = {int(i): pixels(files[int(i)][0]) for i in sorted(set(refs)|set(picks))}\n        low, high = np.percentile(np.concatenate([arrays[int(i)].ravel() for i in refs]), [2, 98])\n        intensity.append({"slot": slot, "p2": float(low), "p98": float(high), "reference_indices": refs.tolist()})\n        for i in picks:\n            file, spacing = files[int(i)]\n            planes.append(crop_plane(arrays[int(i)], spacing, low, high))\n            slot_ids.append(slot)\n            sources.append({"slot": slot, "file": Path(file).name, "series_uid": selection[slot]["series_uid"], "native_index": int(i)})\n    if len(planes) < 3:\n        raise ValueError("Fewer than three acquired slices across the six categories")\n    x = model_ready(planes)\n    ids = np.asarray(slot_ids, dtype=np.int8)\n    triplet_slots = np.stack([ids[:-2], ids[1:-1], ids[2:]], 1)\n    pure = (triplet_slots[:, 0] == triplet_slots[:, 1]) & (triplet_slots[:, 1] == triplet_slots[:, 2])\n    audit = {"study_uid": uid, "recipe_sha256": RECIPE_SHA, "selection": selection,\n             "capacities": capacities, "quotas": quotas.tolist(), "source_slices": len(planes),\n             "windows": len(x), "cross_series_windows": int((~pure).sum()),\n             "source_files": sources, "intensity": intensity, "warnings": warnings}\n    if not np.isfinite(x).all() or tuple(x.shape[1:]) != (3, 384, 384):\n        raise ValueError("Nonfinite or invalid model input")\n    return {"images": x, "triplet_slot_ids": triplet_slots, "same_series": pure,\n            "slot_present": np.asarray([s["present"] for s in selection], bool),\n            "audit_json": np.asarray(json.dumps(audit, allow_nan=False))}, audit\n\n\ndef save_npz(path, data):\n    tmp = path.with_suffix(".npz.tmp")\n    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:\n        for name, array in data.items():\n            with z.open(name+".npy", "w", force_zip64=True) as member:\n                np.lib.format.write_array(member, np.asarray(array), allow_pickle=False)\n    tmp.replace(path)\n\n\ndef find_root():\n    for p in ("/kaggle/input/competitions/rsna-knee-abnormality-detection", "/kaggle/input/rsna-knee-abnormality-detection"):\n        if (Path(p)/"train_series.csv").is_file():\n            return Path(p)\n    raise FileNotFoundError("Attach rsna-knee-abnormality-detection competition data")\n\n\ndef run(shard=0, shards=32, limit=None, root=None, output=None, workers=2):\n    print(json.dumps({\'phase\':\'input_setup\',\'shard\':shard}), flush=True)\n    torch.set_num_threads(1)\n    cv2.setNumThreads(1)\n    root = find_root() if root is None else Path(root)\n    output = Path(output or f"/kaggle/working/six_slot_raptor_shard_{shard:02d}")\n    output.mkdir(parents=True, exist_ok=True)\n    series = pd.read_csv(root/"train_series.csv", dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})\n    labels = pd.read_csv(root/"train.csv", dtype={"StudyInstanceUID": str})\n    ids = sorted(labels.StudyInstanceUID.unique())[shard::shards]\n    if limit:\n        ids = ids[:limit]\n    groups = {str(uid): g.to_dict("records") for uid, g in series.groupby("StudyInstanceUID", sort=False)}\n    series_root = next((root/p for p in ("train_series", "train_images", "train") if (root/p).is_dir()), None)\n    if series_root is None:\n        raise FileNotFoundError("No competition training DICOM directory")\n    json_write(output/"recipe.json", dict(RECIPE, sha256=RECIPE_SHA))\n    labels[labels.StudyInstanceUID.isin(ids)].to_csv(output/"labels_official.csv", index=False)\n    gold = set(labels.loc[labels[LABELS].notna().all(axis=1), "StudyInstanceUID"])\n    # Never recursively scan the competition\'s millions of DICOM files to find\n    # a small annotation CSV; use the two supported attached-dataset layouts.\n    jev = next((p/\'gold74_eval_set.csv\' for p in (\n        Path(\'/kaggle/input/datasets/riadmohamed42/jev-knee-labels\'),\n        Path(\'/kaggle/input/jev-knee-labels\')) if (p/\'gold74_eval_set.csv\').is_file()), None)\n    if jev is None:\n        raise FileNotFoundError(\'Attach riadmohamed42/jev-knee-labels: gold74 metadata is required\')\n    if jev is not None:\n        gold |= set(pd.read_csv(jev, dtype={"StudyInstanceUID": str}).StudyInstanceUID)\n        final2 = jev.parent/"FINAL2_labels_scores.csv"\n        if final2.is_file():\n            table = pd.read_csv(final2, dtype={"StudyInstanceUID": str})\n            table[table.StudyInstanceUID.isin(ids)].to_csv(output/"labels_jev_final2.csv", index=False)\n    pd.DataFrame({"StudyInstanceUID": ids, "split": ["held_out_gold" if uid in gold else "train" for uid in ids]}).to_csv(output/"splits.csv", index=False)\n    print(json.dumps({\'phase\':\'metadata_ready\',\'shard\':shard,\'studies\':len(ids),\'series_root\':str(series_root)}), flush=True)\n    start = time.time()\n    results, failures = [], []\n    def heartbeat(state="running"):\n        value = {"state": state, "shard": shard, "shards": shards, "expected_studies": len(ids),\n                 "completed_studies": len(results), "failed_studies": len(failures),\n                 "elapsed_seconds": round(time.time()-start, 1), "updated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),\n                 "output_bytes": sum(r["bytes"] for r in results), "recipe_sha256": RECIPE_SHA}\n        json_write(output/"progress.json", value)\n        print(json.dumps(value), flush=True)\n    def one(uid):\n        file = output/(uid+".npz")\n        # Reserve worst-case remaining uncompressed bag size before processing.\n        if shutil.disk_usage(output).free < 512*1024**2:\n            raise OSError("Less than 512 MiB free: stop before corrupting output")\n        data, audit = prepare(uid, groups[uid], series_root)\n        save_npz(file, data)\n        # Re-open and check the exact saved model tensor.\n        with np.load(file, allow_pickle=False) as saved:\n            if not np.array_equal(saved["images"], data["images"]):\n                raise ValueError("Saved tensor round-trip mismatch")\n        digest = hashlib.sha256()\n        with file.open("rb") as f:\n            for block in iter(lambda: f.read(8*1024**2), b""):\n                digest.update(block)\n        return {"study_uid": uid, "file": file.name, "windows": audit["windows"],\n                "cross_series_windows": audit["cross_series_windows"], "slot_present": data["slot_present"].tolist(),\n                "bytes": file.stat().st_size, "sha256": digest.hexdigest()}\n    heartbeat()\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        futures = {pool.submit(one, uid): uid for uid in ids}\n        for future in as_completed(futures):\n            uid = futures[future]\n            try:\n                row = future.result()\n                results.append(row)\n                with (output/"index.jsonl").open("a") as f:\n                    f.write(json.dumps(row)+"\\n")\n            except Exception as exc:\n                failure = {"study_uid": uid, "error": f"{type(exc).__name__}: {exc}"}\n                failures.append(failure)\n                with (output/"failures.jsonl").open("a") as f:\n                    f.write(json.dumps(failure)+"\\n")\n                print("STUDY_FAILED "+json.dumps(failure), flush=True)\n            heartbeat()\n    manifest = {"recipe_sha256": RECIPE_SHA, "shard": shard, "shards": shards,\n                "expected_uids": ids, "completed": results, "failures": failures,\n                "complete": len(results) == len(ids) and not failures}\n    json_write(output/"manifest.json", manifest)\n    heartbeat("complete" if manifest["complete"] else "incomplete")\n    if failures:\n        raise RuntimeError(f"Shard incomplete: {len(failures)} failed studies; see failures.jsonl")\n    return manifest\n\n\nif __name__ == "__main__":\n    p = argparse.ArgumentParser()\n    p.add_argument("--shard", type=int, default=0)\n    p.add_argument("--shards", type=int, default=32)\n    p.add_argument("--limit", type=int)\n    p.add_argument("--root")\n    p.add_argument("--output")\n    a = p.parse_args()\n    run(**vars(a))\n', 'infer_fusion_cv5_submission.py': '"""5-fold CV ensemble Kaggle submission: Raptor v5 + RadImageNet fusion.\n\nInternet off. Loads fusion_best.pt of every CV fold (selected per fold by validation accuracy, the mean\nof gold74 and left-out-fold macro AUC) from the attached rsna-fusion-cv5-* kernel outputs. For every\nstudy in sample_submission.csv: DICOM, six-slot preprocessing (exact training recipe), same-series\nwindows, each fold model (fp16, full bag; models alternate across GPUs), sigmoid, mean over folds.\nNo test-time augmentation.\n"""\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom collections import deque\nfrom concurrent.futures import ProcessPoolExecutor\nfrom pathlib import Path\n\nLABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",\n          "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\nTIMM_SHA = \'888981753e65cbaacfc07494370138b1700a27b1f0af587f4f9b47bc024161d0\'\nCHECKPOINT_NAME = \'fusion_best.pt\'   # one per fold\nEVAL_CHUNK = 16\nINPUT = Path(os.environ.get(\'RSNA_INPUT\', \'/kaggle/input\'))\n\n\ndef log(kind, **data):\n    print(json.dumps(dict(event=kind, utc=time.strftime(\'%Y-%m-%dT%H:%M:%SZ\', time.gmtime()), **data)), flush=True)\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for block in iter(lambda: f.read(8*1024**2), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef attached_roots():\n    """Attached datasets/notebook outputs, never the competition DICOM tree (too many files to walk)."""\n    roots = []\n    for base in (INPUT/\'datasets\', INPUT/\'notebooks\'):\n        if base.is_dir():\n            roots += [p for owner in base.iterdir() if owner.is_dir() for p in owner.iterdir() if p.is_dir()]\n    roots += [p for p in INPUT.iterdir() if p.is_dir() and p.name not in\n              (\'datasets\', \'notebooks\', \'competitions\', \'models\', \'rsna-knee-abnormality-detection\')]\n    return roots\n\n\ndef find_one(name, slug):\n    hits = sorted({p for r in attached_roots() if slug in str(r) for p in r.rglob(name)})\n    if len(hits) != 1:\n        raise FileNotFoundError(f\'Expected exactly one {name} under {slug}; found {hits}\')\n    return hits[0]\n\n\ndef ensure_timm():\n    try:\n        version = importlib.metadata.version(\'timm\')\n    except importlib.metadata.PackageNotFoundError:\n        version = None\n    if version != \'1.0.22\' and not os.environ.get(\'RSNA_SKIP_TIMM_INSTALL\'):\n        wheel = find_one(\'timm-1.0.22-py3-none-any.whl\', \'rsna-knee-coatnet-d4-depthzone-swa3-b2\')\n        if sha256(wheel) != TIMM_SHA:\n            raise RuntimeError(\'timm wheel hash mismatch\')\n        subprocess.check_call([sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'--no-deps\', \'--no-index\', str(wheel)])\n        importlib.invalidate_caches()\n    for mod in [m for m in sys.modules if m == \'timm\' or m.startswith(\'timm.\')]:\n        del sys.modules[mod]\n    import timm\n    if timm.__version__ != \'1.0.22\' and not os.environ.get(\'RSNA_SKIP_TIMM_INSTALL\'):\n        raise RuntimeError(f\'timm {timm.__version__} loaded; 1.0.22 required\')\n    return timm.__version__\n\n\ndef competition_root():\n    for p in (INPUT/\'competitions\'/\'rsna-knee-abnormality-detection\', INPUT/\'rsna-knee-abnormality-detection\'):\n        if (p/\'sample_submission.csv\').is_file() and (p/\'test_series.csv\').is_file():\n            return p\n    raise FileNotFoundError(\'Attach the rsna-knee-abnormality-detection competition data\')\n\n\n# ----------------------------------------------------------------------------- preprocessing workers\n_GROUPS, _SERIES_ROOT = None, None\n\n\ndef _init_worker(groups, series_root):\n    global _GROUPS, _SERIES_ROOT\n    import cv2\n    import torch\n    cv2.setNumThreads(1)\n    torch.set_num_threads(1)\n    _GROUPS, _SERIES_ROOT = groups, Path(series_root)\n\n\ndef _prepare(uid):\n    """Returns (uid, float16 [K,3,384,384] same-series windows, n_all_windows, error)."""\n    import numpy as np\n    from preprocess_six_slot_raptor import prepare\n    started = time.time()\n    try:\n        rows = _GROUPS.get(uid)\n        if not rows:\n            raise ValueError(\'No test_series rows for study\')\n        data, audit = prepare(uid, rows, _SERIES_ROOT)\n        images, same = data[\'images\'], np.asarray(data[\'same_series\'], bool)\n        if same.any():              # training/eval convention: windows=\'pure\' with fallback\n            images = images[same]\n        if images.dtype != np.float16 or images.ndim != 4 or images.shape[1:] != (3, 384, 384) or not len(images):\n            raise ValueError(f\'Bad prepared tensor {images.shape} {images.dtype}\')\n        return uid, np.ascontiguousarray(images), int(audit[\'windows\']), None, round(time.time()-started, 2)\n    except Exception as error:   # noqa: BLE001 - recorded and handled by the caller\n        return uid, None, 0, f\'{type(error).__name__}: {str(error)[:300]}\', round(time.time()-started, 2)\n\n\n# ----------------------------------------------------------------------------- checkpoints\ndef fold_checkpoints(folds):\n    """fusion_best.pt of each CV fold, from the attached rsna-fusion-cv5-* kernel outputs."""\n    hits = sorted({p for r in attached_roots() if \'rsna-fusion-cv5-f\' in str(r) for p in r.rglob(CHECKPOINT_NAME)})\n    found = {}\n    for p in hits:\n        tag = [part for part in p.parts if part.startswith(\'fold\') and part[4:].isdigit()]\n        if len(tag) == 1:\n            found.setdefault(int(tag[0][4:]), []).append(p)\n    missing = [k for k in folds if len(found.get(k, [])) != 1]\n    if missing:\n        raise FileNotFoundError(f\'Need exactly one {CHECKPOINT_NAME} per fold; problems for folds {missing}: {found}\')\n    return [found[k][0] for k in folds]\n\n\n# ----------------------------------------------------------------------------- main\ndef main():\n    t0 = time.time()\n    timm_version = ensure_timm()\n    import numpy as np\n    import pandas as pd\n    import torch\n    from models.raptor_v5_radimagenet import load_fused_checkpoint\n\n    folds = [int(k) for k in os.environ.get(\'CV5_FOLDS\', \'0,1,2,3,4\').split(\',\')]\n    budget_h = float(os.environ.get(\'CV5_BUDGET_HOURS\', \'7.5\'))\n    root = competition_root()\n    sample = pd.read_csv(root/\'sample_submission.csv\', dtype={\'StudyInstanceUID\': str})\n    if list(sample.columns) != [\'StudyInstanceUID\', *LABELS]:\n        raise ValueError(f\'Unexpected submission columns: {list(sample.columns)}\')\n    ids = list(sample.StudyInstanceUID)\n    if len(set(ids)) != len(ids):\n        raise ValueError(\'Duplicate study IDs in sample_submission.csv\')\n    series = pd.read_csv(root/\'test_series.csv\', dtype={\'StudyInstanceUID\': str, \'SeriesInstanceUID\': str})\n    groups = {uid: g.to_dict(\'records\') for uid, g in series.groupby(\'StudyInstanceUID\', sort=False)}\n    series_root = next((root/p for p in (\'test_series\', \'test_images\', \'test\') if (root/p).is_dir()), None)\n    if series_root is None:\n        raise FileNotFoundError(\'No test DICOM directory\')\n    workers = max(1, (os.cpu_count() or 2) - 1)\n    log(\'inputs\', studies=len(ids), studies_with_series=sum(u in groups for u in ids), series_root=str(series_root),\n        workers=workers, timm=timm_version, torch=torch.__version__, gpus=torch.cuda.device_count(), folds=folds)\n\n    # Start CPU workers before CUDA is initialised in this process.\n    pool = ProcessPoolExecutor(max_workers=workers, initializer=_init_worker, initargs=(groups, str(series_root)))\n\n    cuda = torch.cuda.is_available()\n    n_gpu = max(1, torch.cuda.device_count()) if cuda else 1\n    models, info = [], []\n    for k, path in zip(folds, fold_checkpoints(folds)):\n        ck = torch.load(path, map_location=\'cpu\', weights_only=True)\n        cfg = ck.get(\'training_config\', {})\n        if cfg.get(\'fold\') != k or ck.get(\'fusion_config\', {}).get(\'rad_input\') != \'pm1_middle\':\n            raise ValueError(f\'{path}: fold {cfg.get("fold")} / rad_input {ck.get("fusion_config", {}).get("rad_input")}\')\n        device = torch.device(\'cuda\', len(models) % n_gpu) if cuda else torch.device(\'cpu\')\n        model = load_fused_checkpoint(ck).to(device).eval()\n        info.append(dict(fold=k, checkpoint=str(path), bytes=path.stat().st_size, selection=ck.get(\'selection\'),\n                         device=str(device)))\n        models.append((model, device))\n        del ck\n    torch.backends.cudnn.benchmark = True\n    log(\'models_loaded\', models=info, parameters_each=sum(p.numel() for p in models[0][0].parameters()))\n\n    def predict(images, use):\n        """Mean sigmoid probability over models. Work for all GPUs is queued before any result is read,\n        so models on different GPUs run concurrently."""\n        x = torch.from_numpy(images)\n        n = len(x)\n        padded = -(-n//EVAL_CHUNK)*EVAL_CHUNK     # fixed chunk shape, as in training evaluation\n        if padded > n:\n            x = torch.cat([x, x[-1:].expand(padded-n, *x.shape[1:])])\n        pinned = x.pin_memory() if cuda else x\n        logits = []\n        for model, device in use:\n            amp = dict(device_type=\'cuda\', dtype=torch.float16) if cuda else dict(device_type=\'cpu\', enabled=False)\n            feats = []\n            for chunk in pinned.split(EVAL_CHUNK):\n                with torch.autocast(**amp):\n                    feats.append(model.backbone(chunk.to(device, non_blocking=True).float()).float())\n            with torch.autocast(**amp):\n                logits.append(model.head(torch.cat(feats, 0)[:n][None]).float())\n        probs = np.stack([torch.sigmoid(l)[0].cpu().numpy() for l in logits])\n        return probs.mean(0), n\n\n    preds, failures, windows, prep_seconds, reduced = {}, {}, {}, [], []\n    done = 0\n    queue, remaining, ahead = deque(), iter(ids), 2*workers + 2\n\n    def submit_next():\n        uid = next(remaining, None)\n        if uid is not None:\n            queue.append(pool.submit(_prepare, uid))\n\n    for _ in range(ahead):\n        submit_next()\n    with torch.inference_mode():\n        while queue:\n            uid, images, n_all, error, seconds = queue.popleft().result()\n            submit_next()\n            done += 1\n            prep_seconds.append(seconds)\n            if error is not None:\n                failures[uid] = error\n                log(\'study_failed\', study=uid, error=error)\n                continue\n            # Time guard: past the budget, use one model per GPU for the remaining studies.\n            use = models if (time.time()-t0)/3600 < budget_h else models[:n_gpu]\n            if len(use) < len(models):\n                reduced.append(uid)\n            p, n = predict(images, use)\n            if not np.isfinite(p).all():\n                failures[uid] = \'non-finite prediction\'\n                continue\n            preds[uid] = p\n            windows[uid] = (n, n_all)\n            if done <= 3 or done % 100 == 0 or done == len(ids):\n                log(\'progress\', done=done, total=len(ids), failed=len(failures), windows_used=n, windows_all=n_all,\n                    models_used=len(use), elapsed_min=round((time.time()-t0)/60, 2))\n    pool.shutdown()\n\n    if not preds:\n        raise RuntimeError(\'No study could be predicted\')\n    # Studies that could not be processed get the mean predicted probability (rank-neutral fallback).\n    fallback = np.mean(np.stack(list(preds.values())), axis=0)\n    rows = [[uid, *(preds.get(uid, fallback)).tolist()] for uid in ids]\n    sub = pd.DataFrame(rows, columns=[\'StudyInstanceUID\', *LABELS])\n    values = sub[LABELS].to_numpy(float)\n    if sub.shape != sample.shape or not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():\n        raise RuntimeError(\'Invalid submission\')\n    sub.to_csv(\'submission.csv\', index=False)\n    receipt = dict(model=f\'Raptor v5 + RadImageNet fusion, {len(models)}-fold CV ensemble (mean of sigmoid \'\n                         \'probabilities, fusion_best.pt per fold, no TTA)\',\n                   folds=info, studies=len(ids), predicted=len(preds), failed=failures,\n                   reduced_ensemble_studies=len(reduced),\n                   mean_windows_used=float(np.mean([w[0] for w in windows.values()])),\n                   prep_seconds_median=float(np.median(prep_seconds)), minutes=round((time.time()-t0)/60, 2))\n    Path(\'inference_receipt.json\').write_text(json.dumps(receipt, indent=2))\n    log(\'submission_written\', **{k: v for k, v in receipt.items() if k not in (\'failed\', \'folds\')},\n        failed=len(failures))\n\n\nif __name__ == \'__main__\':\n    main()\n'}
src = Path('/kaggle/working/fusion_src')
for name, text in SOURCES.items():
    path = src/name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
print('sources written:', sorted(SOURCES), flush=True)


In [ ]:
import os, subprocess, sys
env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONPATH=str(src), OMP_NUM_THREADS='1', CV5_FOLDS='0,1,2,3,4')
proc = subprocess.Popen([sys.executable, str(src/'infer_fusion_cv5_submission.py')], stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, env=env, cwd='/kaggle/working', bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
if proc.wait() != 0:
    raise SystemExit(f'Inference failed with exit code {proc.returncode}')
import pandas as pd
sub = pd.read_csv('/kaggle/working/submission.csv')
print(sub.shape)
print(sub.head())
